# 05 - Events and profiles

Explores the two parquet files produced by the intake step in `out/data/`:

| File | Grain | Columns |
|---|---|---|
| `events.parquet` | one row per event | `user_id`, `ts`, `source`, `fields` (map of string to string) |
| `profiles.parquet` | one row per customer | `user_id`, `as_of`, `attributes` (map), `lifelong` (list of key/timestamp) |

`events.parquet` has ~12.9M rows, so the notebook reads only the columns and row groups it needs instead of loading the whole file.

In [1]:
from pathlib import Path
import json
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

# Works whether the notebook is run from the repo root or from notebooks/
ROOT = Path.cwd()
if not (ROOT / "out" / "data").exists():
    ROOT = ROOT.parent
DATA = ROOT / "out" / "data"
EVENTS, PROFILES = DATA / "events.parquet", DATA / "profiles.parquet"
print(DATA)
for p in sorted(DATA.iterdir()):
    print(f"{p.name:25s} {p.stat().st_size / 1e6:10.1f} MB")

c:\Users\vaijayanti.choudhary\Desktop\FM-Training-Vaijayanti\pragmatiq\out\data
events.parquet                 237.5 MB
output_report.json               0.0 MB
profiles.parquet                 0.3 MB
quality_report.json              0.1 MB


## 1. Intake reports

In [2]:
out_rep = json.loads((DATA / "output_report.json").read_text())
print("generated      :", out_rep["generated"])
print("rows in        :", f"{out_rep['rows_in']:,}")
print("events out     :", f"{out_rep['events_out']:,}")
print("customers      :", f"{out_rep['customers']:,}")
print("profiles out   :", f"{out_rep['profiles_out']:,}")
print("vocab estimate :", f"{out_rep['vocabulary_estimate']:,}")
pd.Series(out_rep["by_source"], name="events").to_frame()

generated      : 2026-11-23T07:08:12+00:00
rows in        : 29,867,531
events out     : 12,918,420
customers      : 8,878
profiles out   : 10,000
vocab estimate : 100,528


,events
app,8813462
transaction,4100074
communication,4884


In [3]:
# Per source table: rows in, events kept, and why rows were dropped
rows = []
for table, r in out_rep["reconciliation"].items():
    rows.append({"table": table, "rows_in": r["rows_in"], "events": r["events"],
                 "dropped": sum(r["dropped"].values()), "unexplained": r["unexplained"],
                 "reasons": "; ".join(f"{k} ({v:,})" for k, v in r["dropped"].items())})
pd.DataFrame(rows).set_index("table")

,rows_in,events,dropped,unexplained,reasons
table,,,,,
collections_case,610,290,320,0,timestamp after the run clock (320)
collections_case_snapshot,4394,2101,2293,0,"timestamp after the run clock (2,293)"
customer_snapshot,57762,24964,32798,0,"timestamp after the run clock (31,799); unchanged since the previous snapsho..."
fraud_case,2171,817,1354,0,"timestamp after the run clock (1,354)"
fraud_case_snapshot,4606,1676,2930,0,"timestamp after the run clock (2,930)"
interaction_event,17413914,8246544,9167370,0,"timestamp after the run clock (9,167,370)"
product_event,360976,132178,228798,0,"timestamp after the run clock (228,798)"
product_snapshot,285532,114612,170920,0,"timestamp after the run clock (165,825); unchanged since the previous snapsh..."
session,847108,409776,437332,0,"timestamp after the run clock (437,332)"


In [4]:
qual = json.loads((DATA / "quality_report.json").read_text())
print(f"client: {qual['client']}   readiness: {qual['readiness']['score']} ({qual['readiness']['band']})")
checks = pd.DataFrame(qual["readiness"]["checks"])
checks["n_findings"] = checks["findings"].str.len()
checks[["name", "weight", "scored", "n_findings"]]

client: 10k_data   readiness: 86.4 (GREEN)


,name,weight,scored,n_findings
0,entity_resolution,20,20.0,0
1,time_validity,20,20.0,0
2,referential_integrity,10,10.0,0
3,uniqueness,10,10.0,0
4,type_conformance,10,10.0,0
5,signal_density,10,6.4,6
6,temporal_sanity,10,0.0,31
7,volume_adequacy,10,10.0,1


In [5]:
for c in qual["readiness"]["checks"]:
    for f in c["findings"]:
        print(f"[{c['name']}] {f}")

[signal_density] collections_case_snapshot: near-empty fields - writeoff_amount 98%, recovery_amount 100%, recovery_source 100%
[signal_density] interaction_event: near-empty fields - error_code 99%, amt_txn 98%
[signal_density] product_event: near-empty fields - old_limit 97%, new_limit 97%, dpd_reset_flag 100%, final_outcome 98%, approved_amount 100%
[signal_density] product_snapshot: near-empty fields - close_date 99%, delinquency_start_date 98%, principal_amount 98%, sanctioned_amount 98%, instalment_amount 98%, instalment_frequency 98% (+1 more)
[signal_density] session: near-empty fields - login_failure_reason 96%
[signal_density] txn_event: near-empty fields - decline_category 98%, is_customer_caused 98%, payment_funding_product 96%, threeds_outcome 96%
[temporal_sanity] collections_case.case_open_timestamp: latest is 2028-09-19 07:21:13.312335 which is in the future (EVENT TIME)
[temporal_sanity] collections_case.promise_made_timestamp: latest is 2028-09-13 00:00:00 which is in

## 2. Events

In [6]:
ev_file = pq.ParquetFile(EVENTS)
print("rows       :", f"{ev_file.metadata.num_rows:,}")
print("row groups :", ev_file.metadata.num_row_groups)
print(ev_file.schema_arrow)

rows       : 12,918,420
row groups : 99
user_id: string not null
ts: timestamp[us] not null
source: string not null
fields: map<string, string ('fields')> not null
  child 0, fields: struct<key: string not null, value: string> not null
      child 0, key: string not null
      child 1, value: string


In [7]:
# A small sample from the first row group. Used for the field-level views below.
sample = ev_file.read_row_group(0).to_pandas()
print(sample.shape)
sample.head()

(131072, 4)


,user_id,ts,source,fields
0,CIN0009920000,2025-10-01,app,"[(age_band, 45-54), (credit_bureau_source, TRANSUNION), (credit_score_bureau..."
1,CIN0009920000,2025-10-01,transaction,"[(currency, USD), (current_balance, 18612.24), (days_delinquent, 0), (instal..."
2,CIN0009920000,2025-10-01,transaction,"[(card_network, VISA), (currency, USD), (current_balance, 0.00), (days_delin..."
3,CIN0009920000,2025-10-01,transaction,"[(card_network, VISA), (credit_limit, 27500.00), (currency, USD), (current_b..."
4,CIN0009920000,2025-10-01,transaction,"[(currency, USD), (current_balance, 51.50), (days_delinquent, 0), (instalmen..."


In [8]:
# Cheap full-file aggregates: read one column at a time
src = ev_file.read(columns=["source"]).column("source").value_counts().to_pylist()
src = pd.DataFrame(src).rename(columns={"values": "source", "counts": "events"}).set_index("source")
src["share"] = (src["events"] / src["events"].sum()).round(4)
src

,events,share
source,,
app,8813462,0.6822
transaction,4100074,0.3174
communication,4884,0.0004


In [9]:
# Time range from the ts column
ts = ev_file.read(columns=["ts"]).column("ts")
print("first event:", pc.min(ts).as_py())
print("last event :", pc.max(ts).as_py())

first event: 2025-06-19 08:48:21.093496
last event : 2026-11-23 07:05:13.739442


In [10]:
# Events per customer
per_user = ev_file.read(columns=["user_id"]).column("user_id").value_counts().to_pylist()
per_user = (pd.DataFrame(per_user).rename(columns={"values": "user_id", "counts": "n_events"})
            .set_index("user_id")["n_events"])
print("customers with events:", f"{len(per_user):,}")
per_user.describe(percentiles=[.01, .1, .5, .9, .99]).round(1)

customers with events: 8,878


count    8878.0
mean     1455.1
std       693.2
min         2.0
1%         39.0
10%       353.7
50%      1604.0
90%      2253.0
99%      2698.2
max      3252.0
Name: n_events, dtype: float64

In [11]:
# Which keys live in the `fields` map, per source (from the sample row group)
def field_keys(df):
    return df["fields"].map(lambda kv: [k for k, _ in kv]).explode().value_counts()

for s, g in sample.groupby("source"):
    keys = field_keys(g)
    print(f"--- {s}: {len(g):,} events, {len(keys)} distinct keys")
    print((keys / len(g)).round(3).head(15).to_string())

--- app: 88,759 events, 43 distinct keys
fields
event_type                0.950
journey_name              0.936
page_or_screen            0.936
product_in_context        0.609
journey_completed_flag    0.060
is_trusted_flag           0.047
login_method              0.047
login_status              0.047
session_channel           0.047
session_end               0.047
device_type               0.045
os                        0.045
app_version               0.031
amt_txn                   0.017
initiated_by              0.014
--- communication: 84 events, 36 distinct keys
fields
case_status                  0.714
amount_collected_to_date     0.548
collection_stage             0.548
dpd_at_action                0.548
action_channel               0.393
action_timestamp             0.393
collections_action_type      0.393
collections_outcome          0.393
contact_outcome              0.393
alert_status                 0.274
total_exposure_amount        0.167
promise_status               0.14

In [12]:
# Look at one source with `fields` expanded into columns
def expand(df):
    wide = pd.DataFrame([dict(kv) for kv in df["fields"]], index=df.index)
    return pd.concat([df[["user_id", "ts", "source"]], wide], axis=1)

expand(sample[sample["source"] == "transaction"].head(10)).dropna(axis=1, how="all")

,user_id,ts,source,currency,current_balance,days_delinquent,instalments_missed_count,open_date,overdue_amount,overdue_principal_amt,premature_withdrawal_flag,product_status,product_tier,product_type,card_network,credit_limit,interest_rate_pct,amount,category,counterparty_type,customer_country_code,debit_credit_indicator,decision,decline_category,event_timestamp,fx_rate,is_customer_caused,is_fraud_flagged,is_recurring,merchant_country,pin_verified_flag,pos_entry_mode,record_type,transaction_type,txn_channel,balance_before,initiator_type,posted_timestamp,posting_status,settlement_timestamp
1,CIN0009920000,2025-10-01 00:00:00.000000,transaction,USD,18612.24,0,0,2016-02-28,0.00,0.00,false,ACTIVE,GOLD,CHECKING_ACCOUNT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,CIN0009920000,2025-10-01 00:00:00.000000,transaction,USD,0.00,0,0,2021-02-14,0.00,0.00,false,DORMANT,GOLD,DEBIT_CARD,VISA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,CIN0009920000,2025-10-01 00:00:00.000000,transaction,USD,0.00,0,0,2024-11-22,0.00,0.00,false,ACTIVE,GOLD,CREDIT_CARD,VISA,27500.00,18.020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,CIN0009920000,2025-10-01 00:00:00.000000,transaction,USD,51.50,0,0,2016-11-23,0.00,0.00,false,ACTIVE,GOLD,DIGITAL_WALLET,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,CIN0009920000,2025-10-02 20:33:30.413495,transaction,USD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.08,DINING,MERCHANT,US,D,DECLINED,EXPIRED_CARD,2025-10-02,1.000000,true,false,false,US,true,CHIP,AUTHORIZATION,PURCHASE,POS,NaN,NaN,NaN,NaN,NaN
6,CIN0009920000,2025-10-03 12:30:18.017267,transaction,USD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,344.29,BILLS_UTILITIES,MERCHANT,US,D,NaN,NaN,2025-10-03,1.000000,NaN,false,true,US,NaN,NaN,POSTING,DIRECT_DEBIT,ONLINE_BANKING,17923.66,CUSTOMER,2025-10-03,POSTED,2025-10-03
7,CIN0009920000,2025-10-03 12:43:37.850591,transaction,USD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.95,SHOPPING,MERCHANT,US,D,APPROVED,NaN,2025-10-03,1.000000,NaN,false,false,US,true,CHIP,AUTHORIZATION,PURCHASE,POS,NaN,NaN,NaN,NaN,NaN
38,CIN0009920000,2025-10-03 19:48:13.278409,transaction,USD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,68.37,GROCERIES,MERCHANT,US,D,APPROVED,NaN,2025-10-03,1.000000,NaN,false,false,US,false,CONTACTLESS,AUTHORIZATION,PURCHASE,POS,NaN,NaN,NaN,NaN,NaN
39,CIN0009920000,2025-10-03 21:44:49.256961,transaction,USD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.00,TRANSFERS,OWN_ACCOUNT,US,C,NaN,NaN,2025-10-03,1.000000,NaN,false,false,NaN,NaN,NaN,POSTING,TRANSFER_IN,MOBILE_APP,51.50,CUSTOMER,2025-10-03,POSTED,2025-10-03
40,CIN0009920000,2025-10-03 21:44:49.256961,transaction,USD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.00,TRANSFERS,OWN_ACCOUNT,US,D,NaN,NaN,2025-10-03,1.000000,NaN,false,false,NaN,NaN,NaN,POSTING,TRANSFER_OUT,MOBILE_APP,17579.37,CUSTOMER,2025-10-03,POSTED,2025-10-03


In [13]:
# One customer's timeline (first 25 events)
uid = sample["user_id"].iloc[0]
tl = pq.read_table(EVENTS, filters=[("user_id", "=", uid)]).to_pandas().sort_values("ts")
print(uid, f"- {len(tl):,} events, {tl['ts'].min()} -> {tl['ts'].max()}")
tl.assign(fields=tl["fields"].map(dict)).head(25)

CIN0009920000 - 2,699 events, 2025-10-01 00:00:00 -> 2026-11-22 23:18:00.352085


,user_id,ts,source,fields
0,CIN0009920000,2025-10-01 00:00:00.000000,app,"{'age_band': '45-54', 'credit_bureau_source': 'TRANSUNION', 'credit_score_bu..."
1,CIN0009920000,2025-10-01 00:00:00.000000,transaction,"{'currency': 'USD', 'current_balance': '18612.24', 'days_delinquent': '0', '..."
2,CIN0009920000,2025-10-01 00:00:00.000000,transaction,"{'card_network': 'VISA', 'currency': 'USD', 'current_balance': '0.00', 'days..."
3,CIN0009920000,2025-10-01 00:00:00.000000,transaction,"{'card_network': 'VISA', 'credit_limit': '27500.00', 'currency': 'USD', 'cur..."
4,CIN0009920000,2025-10-01 00:00:00.000000,transaction,"{'currency': 'USD', 'current_balance': '51.50', 'days_delinquent': '0', 'ins..."
5,CIN0009920000,2025-10-02 20:33:30.413495,transaction,"{'amount': '12.08', 'category': 'DINING', 'counterparty_type': 'MERCHANT', '..."
6,CIN0009920000,2025-10-03 12:30:18.017267,transaction,"{'amount': '344.29', 'balance_before': '17923.66', 'category': 'BILLS_UTILIT..."
7,CIN0009920000,2025-10-03 12:43:37.850591,transaction,"{'amount': '23.95', 'category': 'SHOPPING', 'counterparty_type': 'MERCHANT',..."
8,CIN0009920000,2025-10-03 19:33:40.032913,app,"{'app_version': '7.1.4', 'device_type': 'TABLET', 'is_trusted_flag': 'false'..."
9,CIN0009920000,2025-10-03 19:38:39.835867,app,"{'event_type': 'LOGIN_ATTEMPT', 'journey_name': 'BROWSE', 'page_or_screen': ..."


## 3. Profiles

In [14]:
prof = pq.read_table(PROFILES).to_pandas()
print(prof.shape, "| unique customers:", prof["user_id"].nunique())
print("as_of:", prof["as_of"].min(), "->", prof["as_of"].max())
prof.head()

(10000, 4) | unique customers: 10000
as_of: 2026-11-23 07:05:15.864044 -> 2026-11-23 07:05:15.864044


,user_id,as_of,attributes,lifelong
0,CIN0009920000,2026-11-23 07:05:15.864044,"[(cust_segment, AFFLUENT), (customer_is_active, true), (employment_status, S...","[{'key': 'relationship_open_date', 'ts': 2016-02-28 00:00:00}, {'key': 'acc_..."
1,CIN0009920001,2026-11-23 07:05:15.864044,"[(cust_segment, AFFLUENT), (customer_is_active, true), (home_currency, USD),...","[{'key': 'relationship_open_date', 'ts': 2014-03-09 00:00:00}, {'key': 'acc_..."
2,CIN0009920002,2026-11-23 07:05:15.864044,"[(cust_segment, MASS), (customer_is_active, true), (employment_status, STUDE...","[{'key': 'kyc_last_review_date', 'ts': 2026-06-06 00:00:00}, {'key': 'relati..."
3,CIN0009920003,2026-11-23 07:05:15.864044,"[(cust_segment, MASS), (customer_is_active, true), (employment_status, SELF_...","[{'key': 'relationship_open_date', 'ts': 2019-01-25 00:00:00}, {'key': 'acc_..."
4,CIN0009920004,2026-11-23 07:05:15.864044,"[(cust_segment, MASS), (customer_is_active, true), (employment_status, RETIR...","[{'key': 'relationship_open_date', 'ts': 2022-12-13 00:00:00}, {'key': 'acc_..."


In [15]:
# Expand the `attributes` map into one column per attribute
attrs = pd.DataFrame([dict(kv) for kv in prof["attributes"]], index=prof["user_id"])
print(attrs.shape)
fill = attrs.notna().mean().sort_values(ascending=False).rename("filled").to_frame()
fill["distinct"] = attrs.nunique()
fill

(10000, 14)


,filled,distinct
cust_segment,1.0000,4
customer_is_active,1.0000,2
home_currency,1.0000,6
is_cust_stat_clr_y_n,1.0000,2
marketing_consent,1.0000,2
kyc_status,1.0000,4
residence_country,1.0000,6
residence_state,0.9924,50
employment_status,0.9786,8
kyc_risk_rating,0.9745,3


In [16]:
# Distribution of a few low cardinality attributes
low = [c for c in attrs.columns if attrs[c].nunique() <= 8][:6]
for c in low:
    print(attrs[c].value_counts(dropna=False).to_string(), "\n")

cust_segment
MASS         7198
AFFLUENT     1996
MASS_PLUS     410
PRIVATE       396 

customer_is_active
true     9216
false     784 

employment_status
SALARIED         4678
RETIRED          1748
SELF_EMPLOYED    1366
PART_TIME         692
STUDENT           518
UNEMPLOYED        390
CONTRACT          253
NaN               214
HOMEMAKER         141 

home_currency
USD    9943
CAD      27
GBP      14
MXN       8
AUD       6
EUR       2 

income_band
USD_50K_75K      1989
USD_25K_50K      1766
USD_75K_100K     1432
USD_0_25K        1292
USD_100K_150K    1117
USD_150K_250K     859
USD_250K_500K     689
NaN               547
USD_500K_PLUS     309 

is_cust_stat_clr_y_n
Y    9786
N     214 



In [17]:
# `lifelong` holds dated facts that never change (open dates, KYC reviews, ...)
life = prof[["user_id", "lifelong"]].explode("lifelong").dropna(subset=["lifelong"]).reset_index(drop=True)
life = pd.concat([life["user_id"], pd.json_normalize(life["lifelong"].tolist())], axis=1)
print(life.shape)
life.groupby("key")["ts"].agg(["count", "min", "max"])

(30529, 3)


,count,min,max
key,,,
acc_cre_ts,10000,2003-11-20 10:20:24.034678,2028-09-10 12:03:41.314878
kyc_last_review_date,9745,2006-07-23 00:00:00.000000,2028-09-29 00:00:00.000000
relationship_close_date,784,2006-07-23 00:00:00.000000,2028-09-28 00:00:00.000000
relationship_open_date,10000,2003-11-20 00:00:00.000000,2028-09-10 00:00:00.000000


## 4. Events joined to profiles

In [18]:
# Do the two files cover the same customers?
ev_users, pr_users = set(per_user.index), set(prof["user_id"])
print("customers in events  :", f"{len(ev_users):,}")
print("customers in profiles:", f"{len(pr_users):,}")
print("in both              :", f"{len(ev_users & pr_users):,}")
print("events but no profile:", f"{len(ev_users - pr_users):,}")
print("profile but no events:", f"{len(pr_users - ev_users):,}")

customers in events  : 8,878
customers in profiles: 10,000
in both              : 8,878
events but no profile: 0
profile but no events: 1,122


In [19]:
# Event volume by customer segment, if the attribute exists
if "cust_segment" in attrs.columns:
    j = attrs[["cust_segment"]].join(per_user.rename("n_events")).fillna({"n_events": 0})
    display(j.groupby("cust_segment")["n_events"].agg(["count", "mean", "median", "max"]).round(1))

,count,mean,median,max
cust_segment,,,,
AFFLUENT,1996,1702.5,1880.0,3252.0
MASS,7198,1148.5,1273.5,3105.0
MASS_PLUS,410,1332.7,1411.0,3200.0
PRIVATE,396,1784.6,1857.0,2918.0
